# Eskalatsiya ehtimolini bashorat qilish — Team A3783E69

**Vazifa:** har bir ogohlantirish (signal) uchun uning eskalatsiya qilinish ehtimolini bashorat qilish. Metrika: ROC-AUC.

**Yondashuv qisqacha:**
1. Tranzaksiyalar `signal_id` bo‘yicha agregatsiya qilinadi.
2. EDA shuni ko‘rsatdiki, deyarli har bir signal oldidan keskin faollik (burst) bor. Shuning uchun eskalatsiyani ajratish uchun **burst tarkibi** va uning **mijozning odatiy (baseline) xulqidan farqi** muhim.
3. Feature bloklari: global statistikalar, vaqt oynalari (1/7/30 kun), baseline (31–180 kun), burst va baseline kontrasti, tur×yo‘nalish kesishmasi, tezlik (soniya aniqligida), pass-through (kirimdan keyin tez chiqim), vaqt odatlari.
4. Model: LightGBM (5-fold × 3 seed) va HistGradientBoosting, OOF AUC bo‘yicha blend.

**Ishga tushirish:** ma’lumot fayllari `data/` papkasida bo‘lishi kerak (notebook bilan bir joyda). `Kernel → Restart & Run All`.

## 0. Sozlamalar

In [ ]:
# Kerak bo'lsa: !pip install lightgbm pyarrow
import os, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

TEAM_ID  = "A3783E69"
DATA_DIR = "data"          # nisbiy yo'l: notebook yonidagi data/ papkasi
FIG_DIR  = "figures"       # EDA grafiklari (sayt uchun) shu yerga saqlanadi
SEEDS    = [42, 2024, 7]
N_FOLDS  = 5
os.makedirs(FIG_DIR, exist_ok=True)
np.random.seed(42)

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
print("LightGBM mavjud:", HAS_LGB)

## 1. Ma’lumotlarni yuklash

In [ ]:
train_signals = pd.read_csv(os.path.join(DATA_DIR, "train_signals.csv"))
test_signals  = pd.read_csv(os.path.join(DATA_DIR, "test_signals.csv"))
train_tx = pd.read_parquet(os.path.join(DATA_DIR, "train_transactions.parquet"))
test_tx  = pd.read_parquet(os.path.join(DATA_DIR, "test_transactions.parquet"))

# sample_submission fayl nomi farq qilishi mumkin ("sample_submission (3).csv" va h.k.)
sample_path = sorted(glob.glob(os.path.join(DATA_DIR, "sample_submission*.csv")))[0]
sample_submission = pd.read_csv(sample_path)

for name, df in [("train_signals", train_signals), ("test_signals", test_signals),
                 ("train_tx", train_tx), ("test_tx", test_tx),
                 ("sample_submission", sample_submission)]:
    print(f"{name:18s} {df.shape}")

## 2. Qisqa EDA (grafiklar `figures/` ga saqlanadi)

In [ ]:
y_rate = train_signals["eskalatsiya"].mean()
print(f"Eskalatsiya ulushi: {y_rate:.2%}")
print("Signal sanasi oralig'i (train):", train_signals["signal_sanasi"].min(), "->", train_signals["signal_sanasi"].max())
print("Signal sanasi oralig'i (test): ", test_signals["signal_sanasi"].min(), "->", test_signals["signal_sanasi"].max())

fig, ax = plt.subplots(figsize=(5, 3.5))
train_signals["eskalatsiya"].value_counts().sort_index().plot.bar(ax=ax, color=["#8aa", "#d55"])
ax.set_xticklabels(["Rad etilgan (0)", "Eskalatsiya (1)"], rotation=0)
ax.set_title("Target taqsimoti")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/target_distribution.png", dpi=120); plt.show()

In [ ]:
# Signal oldidagi faollik: har bir kun uchun o'rtacha tranzaksiyalar soni (klass bo'yicha)
_tmp = train_tx[["signal_id", "tranzaksiya_vaqti"]].merge(train_signals, on="signal_id")
_tmp["days_before"] = (pd.to_datetime(_tmp["signal_sanasi"]) - _tmp["tranzaksiya_vaqti"].dt.normalize()).dt.days
n_per_class = train_signals["eskalatsiya"].value_counts()
daily = _tmp.groupby(["eskalatsiya", "days_before"]).size().unstack(0).div(n_per_class, axis=1)

fig, ax = plt.subplots(figsize=(8, 3.5))
daily.plot(ax=ax, color=["#8aa", "#d55"])
ax.set_xlabel("Signal sanasigacha kunlar"); ax.set_ylabel("O'rtacha tranzaksiya / signal")
ax.set_title("Signal oldidagi faollik (burst)"); ax.invert_xaxis(); ax.legend(["Rad etilgan", "Eskalatsiya"])
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/activity_before_signal.png", dpi=120); plt.show()

print("Signal sanasidan keyingi tranzaksiyalar:", (_tmp["days_before"] < 0).sum())
del _tmp

In [ ]:
# Tur va yo'nalish ulushlari klass bo'yicha
_t = train_tx.merge(train_signals[["signal_id", "eskalatsiya"]], on="signal_id")
share_type = pd.crosstab(_t["eskalatsiya"], _t["tranzaksiya_turi"], normalize="index")
share_dir  = pd.crosstab(_t["eskalatsiya"], _t["kirim_chiqim"], normalize="index")
display(share_type.round(4)); display(share_dir.round(4))

fig, ax = plt.subplots(figsize=(7, 3.5))
for cls, col in [(0, "#8aa"), (1, "#d55")]:
    ax.hist(_t.loc[_t["eskalatsiya"] == cls, "miqdor_indeksi"], bins=80, density=True,
            alpha=0.5, color=col, label=["Rad etilgan", "Eskalatsiya"][cls])
ax.set_title("miqdor_indeksi taqsimoti"); ax.legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/amount_distribution.png", dpi=120); plt.show()
del _t

## 3. Feature engineering

Bitta `build_features(signals, tx)` funksiyasi train va test uchun **bir xil** ishlatiladi (mos kelmaslik xavfi yo‘q).

Oynalar signal sanasiga nisbatan: `w1` (0–1 kun), `w7`, `w30`, `base` (31–180 kun = mijozning odatiy xulqi).

In [ ]:
TYPES = ["karta", "bank_otkazmasi", "naqd", "xalqaro"]
DIRS  = ["kirim", "chiqim"]
TD    = [f"{t}_{d}" for t in TYPES for d in DIRS]
WINDOWS = {"w1": (0, 1), "w7": (0, 7), "w30": (0, 30), "base": (31, 10_000)}


def prepare_tx(signals, tx):
    s = signals[["signal_id", "signal_sanasi"]].copy()
    s["signal_sanasi"] = pd.to_datetime(s["signal_sanasi"])
    t = tx.merge(s, on="signal_id", how="inner")
    t["tranzaksiya_vaqti"] = pd.to_datetime(t["tranzaksiya_vaqti"])
    t["days_before"] = (t["signal_sanasi"] - t["tranzaksiya_vaqti"].dt.normalize()).dt.days.astype("int32")
    t["td"] = t["tranzaksiya_turi"].astype(str) + "_" + t["kirim_chiqim"].astype(str)
    t["amt"] = t["miqdor_indeksi"].astype("float32")
    t = t.sort_values(["signal_id", "tranzaksiya_vaqti"], kind="mergesort").reset_index(drop=True)
    return t


def block_features(d, p):
    """Bitta oyna uchun: son, miqdor statistikasi, tur/yo'nalish/kesishma ulushlari."""
    g = d.groupby("signal_id")["amt"]
    out = pd.DataFrame({
        f"{p}_cnt": g.size(), f"{p}_amt_mean": g.mean(), f"{p}_amt_std": g.std(),
        f"{p}_amt_max": g.max(), f"{p}_amt_min": g.min(), f"{p}_amt_sum": g.sum(),
        f"{p}_amt_med": g.median(),
        f"{p}_active_days": d.groupby("signal_id")["days_before"].nunique(),
    })
    for col, vals in [("tranzaksiya_turi", TYPES), ("kirim_chiqim", DIRS), ("td", TD)]:
        cnt = d.groupby(["signal_id", col]).size().unstack(fill_value=0).reindex(columns=vals, fill_value=0)
        share = cnt.div(cnt.sum(axis=1), axis=0)
        share.columns = [f"{p}_share_{v}" for v in vals]
        out = out.join(share)
    # kesishma bo'yicha o'rtacha miqdor (masalan naqd_chiqim)
    m = d.groupby(["signal_id", "td"])["amt"].mean().unstack().reindex(columns=TD)
    m.columns = [f"{p}_amtmean_{v}" for v in TD]
    out = out.join(m)
    # kirim-chiqim balansi
    sd = d.groupby(["signal_id", "kirim_chiqim"])["amt"].sum().unstack(fill_value=0).reindex(columns=DIRS, fill_value=0)
    out[f"{p}_net_kirim_minus_chiqim_cnt"] = (
        d.groupby(["signal_id", "kirim_chiqim"]).size().unstack(fill_value=0)
         .reindex(columns=DIRS, fill_value=0).pipe(lambda x: x["kirim"] - x["chiqim"]))
    out[f"{p}_amt_sum_kirim"] = sd["kirim"]; out[f"{p}_amt_sum_chiqim"] = sd["chiqim"]
    return out


def velocity_features(t):
    """Soniya aniqligidagi tezlik va pass-through feature'lari."""
    same = t["signal_id"].eq(t["signal_id"].shift())
    dt = t["tranzaksiya_vaqti"].diff().dt.total_seconds().where(same)
    t = t.assign(dt=dt, log_dt=np.log1p(dt.clip(lower=0)))
    g = t.groupby("signal_id")
    out = pd.DataFrame({
        "gap_min_sec": g["dt"].min(), "gap_med_sec": g["dt"].median(),
        "gap_logmean": g["log_dt"].mean(), "gap_logstd": g["log_dt"].std(),
    })
    valid = t["dt"].notna()
    lt = pd.DataFrame({"signal_id": t["signal_id"],
                       "lt60": (t["dt"] < 60).astype(float).where(valid),
                       "lt1h": (t["dt"] < 3600).astype(float).where(valid)}).groupby("signal_id").mean()
    out["gap_lt_60s_share"] = lt["lt60"]; out["gap_lt_1h_share"] = lt["lt1h"]
    hour_bucket = t["tranzaksiya_vaqti"].dt.floor("h")
    out["max_tx_per_hour"] = t.groupby([t["signal_id"], hour_bucket]).size().groupby(level=0).max()
    out["max_tx_per_day"] = t.groupby(["signal_id", "days_before"]).size().groupby(level=0).max()

    # Pass-through: chiqimdan oldingi eng yaqin kirim qancha vaqt oldin bo'lgan
    is_in = t["kirim_chiqim"].eq("kirim")
    last_in_time = t["tranzaksiya_vaqti"].where(is_in).groupby(t["signal_id"]).ffill()
    last_in_amt = t["amt"].where(is_in).groupby(t["signal_id"]).ffill()
    is_out = t["kirim_chiqim"].eq("chiqim")
    lag_h = (t["tranzaksiya_vaqti"] - last_in_time).dt.total_seconds() / 3600
    pt24 = (is_out & (lag_h <= 24)).astype(float)
    pt1 = (is_out & (lag_h <= 1)).astype(float)
    pt_sim = (is_out & (lag_h <= 24) & ((t["amt"] - last_in_amt).abs() < 0.25)).astype(float)
    tmp = pd.DataFrame({"signal_id": t["signal_id"], "is_out": is_out.astype(float),
                        "pt24": pt24, "pt1": pt1, "pt_sim": pt_sim, "w7": t["days_before"] <= 7})
    for scope, sub in [("all", tmp), ("w7", tmp[tmp["w7"]])]:
        gg = sub.groupby("signal_id")[["is_out", "pt24", "pt1", "pt_sim"]].sum()
        den = gg["is_out"].replace(0, np.nan)
        out[f"{scope}_passthru24_share"] = gg["pt24"] / den
        out[f"{scope}_passthru1h_share"] = gg["pt1"] / den
        out[f"{scope}_passthru_similar_share"] = gg["pt_sim"] / den
        out[f"{scope}_passthru24_cnt"] = gg["pt24"]
    return out


def habit_features(t):
    h = t["tranzaksiya_vaqti"].dt.hour
    wd = t["tranzaksiya_vaqti"].dt.weekday
    tmp = pd.DataFrame({"signal_id": t["signal_id"], "night": ((h < 6) | (h >= 22)).astype(float),
                        "weekend": (wd >= 5).astype(float), "hour": h.astype(float),
                        "w7": t["days_before"] <= 7})
    out = tmp.groupby("signal_id")[["night", "weekend", "hour"]].agg(["mean"])
    out.columns = ["night_share", "weekend_share", "hour_mean"]
    w7 = tmp[tmp["w7"]].groupby("signal_id")[["night", "weekend"]].mean()
    out["w7_night_share"] = w7["night"]; out["w7_weekend_share"] = w7["weekend"]
    g = t.groupby("signal_id")["days_before"]
    out["days_since_last_tx"] = g.min(); out["days_since_first_tx"] = g.max()
    out["history_span_days"] = g.max() - g.min()
    return out


def build_features(signals, tx):
    t0 = time.time()
    t = prepare_tx(signals, tx)
    parts = [block_features(t, "all")]
    for name, (lo, hi) in WINDOWS.items():
        sub = t[(t["days_before"] >= lo) & (t["days_before"] <= hi)]
        parts.append(block_features(sub, name))
    parts += [velocity_features(t), habit_features(t)]
    F = pd.concat(parts, axis=1)

    # Burst va baseline kontrasti
    base_days = (F["history_span_days"].clip(upper=180) - 30).clip(lower=1)
    base_rate = F["base_cnt"].fillna(0) / base_days
    for w, n in [("w1", 2), ("w7", 8), ("w30", 31)]:
        F[f"{w}_rate_per_day"] = F[f"{w}_cnt"].fillna(0) / n
        F[f"{w}_rate_vs_base"] = F[f"{w}_rate_per_day"] / (base_rate + 0.1)
        F[f"{w}_cnt_share_of_all"] = F[f"{w}_cnt"].fillna(0) / F["all_cnt"]
    for w in ["w1", "w7", "w30"]:
        F[f"{w}_amt_z_vs_base"] = (F[f"{w}_amt_mean"] - F["base_amt_mean"]) / (F["base_amt_std"] + 0.05)
        F[f"{w}_amtmax_z_vs_base"] = (F[f"{w}_amt_max"] - F["base_amt_mean"]) / (F["base_amt_std"] + 0.05)
        for v in TYPES + DIRS + TD:
            F[f"{w}_dshare_{v}"] = F[f"{w}_share_{v}"] - F[f"base_share_{v}"]

    # Signal sanasi
    s = signals.set_index("signal_id")
    sd = pd.to_datetime(s["signal_sanasi"])
    F["signal_month"] = sd.dt.month
    F["signal_weekday"] = sd.dt.weekday

    F = F.reindex(signals["signal_id"])        # signal tartibini saqlaymiz (tranzaksiyasi yo'q signal ham qoladi)
    F = F.replace([np.inf, -np.inf], np.nan).astype("float32")
    print(f"Feature'lar: {F.shape}  ({time.time() - t0:.0f} s)")
    return F

In [ ]:
X_train = build_features(train_signals, train_tx)
X_test  = build_features(test_signals, test_tx)
X_test  = X_test.reindex(columns=X_train.columns)   # ustunlar bir xil bo'lishi kafolati
y = train_signals.set_index("signal_id").loc[X_train.index, "eskalatsiya"].values

assert list(X_train.columns) == list(X_test.columns)
assert (X_train.index == train_signals["signal_id"].values).all()
assert (X_test.index == test_signals["signal_id"].values).all()
print("Train:", X_train.shape, " Test:", X_test.shape, " Eskalatsiya ulushi:", y.mean().round(4))

## 4. Modellash

Bir xil fold'larda LightGBM va HistGradientBoosting o‘qitiladi. Har bir model uchun OOF (out-of-fold) AUC hisoblanadi, test prognozlari fold va seed'lar bo‘yicha o‘rtachalanadi.

In [ ]:
LGB_PARAMS = dict(
    objective="binary", learning_rate=0.02, num_leaves=15, min_child_samples=40,
    feature_fraction=0.5, bagging_fraction=0.8, bagging_freq=1,
    lambda_l2=5.0, n_estimators=3000, verbose=-1,
)

def run_cv(model_name):
    oof = np.zeros(len(X_train)); test_pred = np.zeros(len(X_test))
    importances = pd.Series(0.0, index=X_train.columns)
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for tr, va in skf.split(X_train, y):
            Xtr, Xva, ytr, yva = X_train.iloc[tr], X_train.iloc[va], y[tr], y[va]
            if model_name == "lgb":
                m = lgb.LGBMClassifier(**LGB_PARAMS, random_state=seed)
                m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="auc",
                      callbacks=[lgb.early_stopping(200, verbose=False)])
                importances += pd.Series(m.booster_.feature_importance("gain"), index=X_train.columns)
            else:
                m = HistGradientBoostingClassifier(
                    learning_rate=0.03, max_iter=2000, max_leaf_nodes=15, min_samples_leaf=40,
                    l2_regularization=5.0, early_stopping=True, validation_fraction=0.15,
                    n_iter_no_change=100, random_state=seed)
                m.fit(Xtr, ytr)
            oof[va] += m.predict_proba(Xva)[:, 1] / len(SEEDS)
            test_pred += m.predict_proba(X_test)[:, 1] / (len(SEEDS) * N_FOLDS)
    auc = roc_auc_score(y, oof)
    print(f"{model_name}: OOF ROC-AUC = {auc:.4f}")
    return oof, test_pred, importances

results = {}
if HAS_LGB:
    results["lgb"] = run_cv("lgb")
results["hgb"] = run_cv("hgb")

In [ ]:
# Blend: OOF bo'yicha eng yaxshi vaznni tanlaymiz (faqat 3 ta variant, CV'ga ortiqcha moslashmaslik uchun)
if "lgb" in results:
    best_w, best_auc = None, -1
    for w in [1.0, 0.7, 0.5]:
        auc = roc_auc_score(y, w * results["lgb"][0] + (1 - w) * results["hgb"][0])
        print(f"w_lgb={w:.1f}  OOF AUC={auc:.4f}")
        if auc > best_auc:
            best_w, best_auc = w, auc
    final_oof  = best_w * results["lgb"][0] + (1 - best_w) * results["hgb"][0]
    final_test = best_w * results["lgb"][1] + (1 - best_w) * results["hgb"][1]
else:
    best_w, final_oof, final_test = 0.0, results["hgb"][0], results["hgb"][1]
print(f"\nYakuniy OOF ROC-AUC: {roc_auc_score(y, final_oof):.4f}  (w_lgb={best_w})")

In [ ]:
# Kalibratsiya tekshiruvi: OOF prognoz decile'lari bo'yicha haqiqiy eskalatsiya ulushi
dec = pd.qcut(final_oof, 10, labels=False, duplicates="drop")
calib = pd.DataFrame({"decile": dec, "y": y, "p": final_oof}).groupby("decile").agg(
    real_rate=("y", "mean"), mean_pred=("p", "mean"), n=("y", "size"))
display(calib.round(3))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(calib.index, calib["real_rate"], color="#d55")
ax.axhline(y.mean(), ls="--", color="gray", label="o'rtacha")
ax.set_xlabel("Prognoz decile (0 = eng past)"); ax.set_ylabel("Haqiqiy eskalatsiya ulushi")
ax.set_title("Model decile'lari bo'yicha eskalatsiya"); ax.legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/decile_lift.png", dpi=120); plt.show()

In [ ]:
if "lgb" in results:
    imp = results["lgb"][2].sort_values(ascending=False)
    imp = imp / imp.sum()
    display(imp.head(30).round(4))
    fig, ax = plt.subplots(figsize=(7, 7))
    imp.head(25)[::-1].plot.barh(ax=ax, color="#d55")
    ax.set_title("Top-25 feature importance (LightGBM gain)")
    plt.tight_layout(); plt.savefig(f"{FIG_DIR}/feature_importance.png", dpi=120); plt.show()
    imp.to_csv(f"{FIG_DIR}/feature_importance.csv")

## 5. Submission (ID bo‘yicha merge — tartib xatosi bo‘lishi mumkin emas)

In [ ]:
pred_df = pd.DataFrame({"signal_id": X_test.index, "ehtimollik": np.clip(final_test, 0, 1)})
submission = sample_submission[["signal_id"]].merge(pred_df, on="signal_id", how="left")

# Qoidalar tekshiruvi
test_ids = set(test_signals["signal_id"])
assert len(submission) == len(test_signals), "Qatorlar soni mos emas"
assert submission["signal_id"].is_unique, "Dublikat ID bor"
assert set(submission["signal_id"]) == test_ids, "ID to'plami mos emas"
assert submission["ehtimollik"].notna().all(), "NaN prognoz bor"
assert submission["ehtimollik"].between(0, 1).all(), "0..1 oralig'idan tashqari qiymat"
assert list(submission.columns) == ["signal_id", "ehtimollik"]

out_path = f"team_{TEAM_ID}.csv"
submission.to_csv(out_path, index=False)
print("Saqlandi:", out_path, submission.shape)
print(submission["ehtimollik"].describe().round(4))
display(submission.head())

## 6. Xulosa

- Target nomutanosib: ~17% eskalatsiya. Metrika ROC-AUC bo‘lgani uchun klass vaznlari shart emas.
- Deyarli har bir signal oldidan tranzaksiyalar keskin ko‘payadi (burst), shuning uchun faollikning o‘zi emas, **burst tarkibining baseline’dan farqi** asosiy signal manbai.
- Eng foydali feature’lar yuqoridagi importance jadvalida ko‘rsatilgan.
- Barcha random seed’lar belgilangan; `Restart & Run All` bir xil natija beradi.